# 🗣️ VibeVoice (Hindi 1.5B) — Zero-Shot Voice Cloning

Clone a voice from a short reference clip and make it speak a hard-coded **Hindi**
sentence, using **`tarun7r/vibevoice-hindi-1.5B`** — a Hindi fine-tune of
Microsoft's **VibeVoice** (a 1.5B next-token-diffusion TTS model with a
Qwen2.5-1.5B LLM backbone). Output is **24 kHz**.

**How it works (the honest version).** Unlike Orpheus (a token-completion
Speech-LLM that needs SNAC codec tokens and a hand-built two-turn prompt) and
like VoxCPM2, VibeVoice exposes a **high-level API**. You hand it a *script*
(text labelled `Speaker 1:`) plus one or more **reference voice clips**. The
model encodes each reference internally (its own continuous speech tokenizer at
7.5 Hz) and synthesizes the script in that voice. **No reference transcript is
needed** — conditioning is on the audio directly — so there is no `REF_TRANSCRIPT`
field like Orpheus has.

VibeVoice's headline feature is **long-form, multi-speaker** synthesis (podcasts,
dialogues, up to 4 speakers). Single-speaker cloning — the focus of this
notebook — is just the 1-speaker case. A 2-speaker Hindi dialogue example is in
the optional extras cell at the bottom.

> ⚠️ **Limitations — read these.** Clone quality depends on a **clean** reference
> (single speaker, no music/noise). VibeVoice clones well from ~10–30 s clips;
> longer is not always better. Because this is a **Hindi** fine-tune, the
> `TARGET_TEXT` should be in **Hindi (Devanagari)** — English text will read
> poorly. Expect to try a couple of reference clips / seeds.

> ℹ️ **Install note.** Microsoft removed the original VibeVoice code from its repo;
> this notebook installs the maintained **community fork**
> (`vibevoice-community/VibeVoice`), which is what the Hindi model card targets.

---
## ✋ Consent & responsible use
Only clone voices you **own** or have **explicit permission** to clone. Do not
use this to impersonate, deceive, or create misleading content. Cloning a
person's voice without consent may be illegal in your jurisdiction. VibeVoice is
released for **research** use and is explicitly **not** licensed for voice
impersonation without recorded consent.

---
## ▶️ Running this in Google Colab
1. **Runtime → Change runtime type → GPU**. ~8 GB VRAM is enough (a T4 works; an
   Ampere+ GPU like L4/A100 is faster). CPU/`mps` work but are slow.
2. Get the notebook into Colab — any one of:
   - **File → Upload notebook** and pick this `.ipynb`, **or**
   - **File → Open notebook → GitHub** tab, paste your repo URL, **or**
   - clone the repo from a cell: `!git clone https://github.com/<you>/<repo>.git`
     then open the `.ipynb` from the file browser on the left.
3. Run the cells **top to bottom**. The first install + model download (~11 GB)
   take several minutes.

## 1. Setup and installation
VibeVoice is **not** on PyPI. We `git clone` the community fork and install it
in editable mode (this also pulls a compatible torch/transformers stack), plus
audio I/O libraries. Safe to re-run. On Colab `ffmpeg` is preinstalled (needed
for MP3/M4A decoding).

> Note: we install with plain `pip` (Colab has no `uv`) and pass the repo's
> **absolute path**, because a `cd` inside a subprocess does not persist.

In [ ]:
# --- Install dependencies (Colab or fresh local env) ---------------------
# %%capture  # uncomment to hide pip noise
import importlib, sys, os, subprocess, shutil

def run(*cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(list(cmd), check=check)

def pip_install(*pkgs):
    run(sys.executable, "-m", "pip", "install", "-q", *pkgs)

# Clone the maintained community fork (the original MS repo was taken down).
REPO_URL = "https://github.com/vibevoice-community/VibeVoice.git"
REPO_DIR = "/content/VibeVoice" if os.path.isdir("/content") else os.path.abspath("VibeVoice")

if not os.path.isdir(REPO_DIR):
    run("git", "clone", "--depth", "1", REPO_URL, REPO_DIR)
else:
    print("✓ Repo already present at", REPO_DIR)

# Editable install puts the `vibevoice` package on the import path regardless of cwd.
# NOTE: default install does NOT compile flash-attn; we use the 'sdpa' attention
# backend below, which needs no extra build and runs on any GPU (incl. T4/Turing).
pip_install("-e", REPO_DIR)

# Jupyter kernels do not always pick up fresh editable installs immediately.
# Add the cloned repo itself to sys.path so `import vibevoice` works reliably
# in the current kernel without requiring a restart.
for candidate in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
# Audio I/O (soundfile writes/reads WAV; librosa decodes/preps the reference).
pip_install("soundfile", "librosa")

if shutil.which("ffmpeg") is None:
    print("⚠️ ffmpeg not found. WAV will still work, but MP3/M4A may not.")
    print("   Colab: run  !apt-get -y install ffmpeg")
    print("   macOS: brew install ffmpeg   |   Ubuntu: sudo apt install ffmpeg")
else:
    print("✅ ffmpeg found:", shutil.which("ffmpeg"))

importlib.invalidate_caches()
vv = importlib.import_module("vibevoice")
print("✅ vibevoice importable from:", vv.__file__)
print("✅ Install step finished. Repo:", REPO_DIR)

## 2. Imports & device
VibeVoice runs on CUDA, Apple `mps`, or CPU. We do **not** hard-require a GPU —
but one is strongly recommended (CPU synthesis is very slow). We also import the
VibeVoice classes here so any install problem surfaces early.

In [ ]:
import gc, importlib, os, subprocess, sys, warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")

REPO_DIR = "/content/VibeVoice" if os.path.isdir("/content") else os.path.abspath("VibeVoice")
for candidate in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

try:
    importlib.import_module("vibevoice")
except ModuleNotFoundError:
    print("`vibevoice` was not on the active kernel path. Reinstalling editable package...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    importlib.invalidate_caches()
    importlib.import_module("vibevoice")

# --- VibeVoice classes (from the community fork installed above) -------------
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

# --- Pick the best available device ------------------------------------------
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("   VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Apple MPS device (no CUDA). Synthesis will be slower than a GPU.")
else:
    DEVICE = "cpu"
    print("⚠️ No GPU/MPS detected — running on CPU. This will be SLOW.")
    print("   In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU.")

print("   torch:", torch.__version__, "| device:", DEVICE)

## 3. Configuration
Everything you might change lives here.

**Set `TARGET_TEXT` to Hindi (Devanagari).** This is a Hindi fine-tune — English
text will read poorly. There is **no reference-transcript field**: VibeVoice
conditions on the reference *audio* directly.

In [ ]:
# --- Model -------------------------------------------------------------------
MODEL_ID      = "tarun7r/vibevoice-hindi-1.5B"  # Hindi fine-tune (weights)
BASE_MODEL_ID = "vibevoice/VibeVoice-1.5B"      # base repo — holds the processor/
                                                # tokenizer config the Hindi repo
                                                # omits (used as a fallback below)
ATTN_IMPLEMENTATION = "sdpa"   # 'sdpa' works everywhere (incl. T4). Use
                               # 'flash_attention_2' only on Ampere+ GPUs that
                               # have flash-attn built (faster, optional).

# --- Reference voice ---------------------------------------------------------
REF_AUDIO_PATH = ""     # leave "" to upload in Colab, or set a local path here
# NOTE: unlike Orpheus, VibeVoice needs NO transcript of the reference clip.

# --- What the cloned voice should say (hard-coded, per requirements) ----------
# Hindi (Devanagari). Edit freely — keep it in Hindi for best results.
TARGET_TEXT = (
    "नमस्ते! यह VibeVoice का उपयोग करके एक क्लोन की गई आवाज़ का परीक्षण है। "
    "मॉडल को इस वाक्य को अपलोड किए गए आवाज़ नमूने की शैली में बोलना चाहिए।"
)

# --- Generation knobs --------------------------------------------------------
CFG_SCALE      = 1.3     # classifier-free guidance. Higher = follows the
                         # voice/prompt more strongly (try 1.3–1.6).
MAX_NEW_TOKENS = None    # None = let the model stop on its own (recommended).
SEED           = 42      # fixed seed for reproducibility; change to vary output.
TORCH_DTYPE    = torch.bfloat16   # on CUDA. bf16 is numerically stable for the
                                  # diffusion head and runs on all GPUs (slower
                                  # on pre-Ampere). CPU/mps force float32 below.

# --- Audio preprocessing (light — the processor also resamples internally) ----
SAMPLE_RATE    = 24000   # VibeVoice operates at / outputs 24 kHz
TRIM_TOP_DB    = 30      # silence-trim aggressiveness (higher = less trimming)
PEAK_NORM      = 0.95    # safe peak normalization target
IDEAL_MIN_SEC  = 10      # VibeVoice clones well from ~10–30 s
IDEAL_MAX_SEC  = 30

OUTPUT_PATH = "output_cloned_voice.wav"
print("Config loaded. Target (Hindi) text:\n ", TARGET_TEXT)

## 4. Upload / load voice sample
In Colab a file picker appears. Locally (or to skip the picker), set
`REF_AUDIO_PATH` in the config cell. Accepts WAV/MP3/M4A/FLAC.

In [ ]:
# Detect Colab and offer an upload widget if no path was configured.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not REF_AUDIO_PATH:
    if IN_COLAB:
        print("Upload your voice sample (WAV/MP3/M4A/FLAC, ~10-30s):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        REF_AUDIO_PATH = list(uploaded.keys())[0]
    else:
        raise RuntimeError(
            "REF_AUDIO_PATH is empty and not running in Colab.\n"
            "Set REF_AUDIO_PATH in the Configuration cell to your audio file."
        )

if not os.path.exists(REF_AUDIO_PATH):
    raise FileNotFoundError(f"Audio file not found: {REF_AUDIO_PATH}")
print("✅ Reference audio:", REF_AUDIO_PATH)

## 5. Audio preprocessing (light)
The processor loads and resamples the reference internally, so heavy prep isn't
required. We do a light cleanup anyway — convert to **mono @ 24 kHz**, trim
leading/trailing silence, peak-normalize — and write a clean WAV that we pass to
the model. This also lets us validate the duration and preview the clip.

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display

def load_and_preprocess(path):
    """Return a cleaned float32 mono waveform at SAMPLE_RATE."""
    try:
        wav, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not decode '{path}'. Unsupported format or missing ffmpeg.\n"
            f"Original error: {e}"
        )
    # Trim silence from both ends.
    wav, _ = librosa.effects.trim(wav, top_db=TRIM_TOP_DB)
    # Safe peak normalization (avoid divide-by-zero on silent clips).
    peak = float(np.max(np.abs(wav))) if wav.size else 0.0
    if peak > 0:
        wav = (wav / peak) * PEAK_NORM
    return wav.astype(np.float32), SAMPLE_RATE

ref_wav, ref_sr = load_and_preprocess(REF_AUDIO_PATH)
duration = len(ref_wav) / ref_sr

# Save the cleaned reference; this is the path we hand to VibeVoice.
CLEAN_REF_PATH = "reference_clean.wav"
sf.write(CLEAN_REF_PATH, ref_wav, ref_sr)
print(f"Processed: {duration:.1f}s, mono, {ref_sr} Hz -> {CLEAN_REF_PATH}")

if duration < 3:
    print(f"🚨 WARNING: only {duration:.1f}s — very short. Cloning will be poor.")
elif duration < IDEAL_MIN_SEC:
    print(f"⚠️ {duration:.1f}s is a little short; {IDEAL_MIN_SEC}-{IDEAL_MAX_SEC}s is ideal.")
elif duration > IDEAL_MAX_SEC:
    print(f"⚠️ {duration:.1f}s is longer than {IDEAL_MAX_SEC}s; fine, but not necessary.")
else:
    print("✅ Duration is in a good range for cloning.")

print("Preview of your processed reference:")
display(Audio(ref_wav, rate=ref_sr))

## 6. Model loading
Loads the VibeVoice **processor** and **model**. Two important details:

- The Hindi repo ships **weights only** (no tokenizer/processor config), so we
  load the processor from `MODEL_ID` first and **fall back to the base repo**
  `vibevoice/VibeVoice-1.5B` if that fails.
- Device-specific dtype/attention: bf16 + `sdpa` on CUDA, float32 + `sdpa` on
  CPU/`mps`. Includes an out-of-memory hint.

The first run downloads ~11 GB of weights (a few minutes).

In [ ]:
# --- Processor (try Hindi repo, fall back to base which holds the configs) ----
try:
    processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
    print(f"✅ Processor loaded from {MODEL_ID}")
except Exception as e:
    print(f"ℹ️ Processor not available in {MODEL_ID} ({type(e).__name__}); "
          f"falling back to base {BASE_MODEL_ID}")
    processor = VibeVoiceProcessor.from_pretrained(BASE_MODEL_ID)
    print(f"✅ Processor loaded from {BASE_MODEL_ID}")

# --- Model (weights from the Hindi repo) -------------------------------------
try:
    if DEVICE == "cuda":
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_ID,
            torch_dtype=TORCH_DTYPE,
            device_map="cuda",
            attn_implementation=ATTN_IMPLEMENTATION,
        )
    else:  # cpu / mps -> float32 + sdpa, then move
        model = VibeVoiceForConditionalGenerationInference.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float32,
            attn_implementation="sdpa",
        )
        model.to(DEVICE)
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); gc.collect()
    raise RuntimeError(
        "CUDA OOM while loading VibeVoice.\n"
        "Fixes: restart the runtime, or use a larger GPU (~8 GB needed)."
    )
except Exception as e:
    raise RuntimeError(
        f"Model download/load failed: {e}\n"
        "If the model is gated, run `huggingface-cli login` (or set HF_TOKEN) and "
        "accept the model terms on its Hugging Face page. If you saw an attention\n"
        "error, set ATTN_IMPLEMENTATION='sdpa' in the config cell."
    )

model.eval()
print(f"✅ VibeVoice (Hindi) ready on {DEVICE}. Output sample rate: {SAMPLE_RATE} Hz")

## 7. Voice cloning / generation
We build a one-line **script** in VibeVoice's format — `Speaker 1: <text>` (speaker
labels are **1-indexed**) — and pass the reference clip as `voice_samples`. The
list order maps `Speaker 1` → first voice clip. The processor encodes the
reference internally; no transcript is needed.

`model.generate(...)` returns `speech_outputs`, a list of audio tensors at
24 kHz.

In [ ]:
# Reproducibility (sampling/diffusion noise).
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# VibeVoice script format: one line per turn, labelled 'Speaker N:' (1-indexed).
# Single speaker -> a single 'Speaker 1:' line. voice_samples[0] is that speaker.
full_script = f"Speaker 1: {TARGET_TEXT}"

inputs = processor(
    text=[full_script],
    voice_samples=[[CLEAN_REF_PATH]],   # one batch item, one speaker -> one clip
    padding=True,
    return_tensors="pt",
    return_attention_mask=True,
)

# Move tensors to the model's device (BatchFeature.to, with a dict fallback).
try:
    inputs = inputs.to(DEVICE)
except Exception:
    inputs = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in inputs.items()}

try:
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            cfg_scale=CFG_SCALE,
            tokenizer=processor.tokenizer,
            generation_config={"do_sample": False},
            verbose=True,
        )
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache(); gc.collect()
    raise RuntimeError(
        "CUDA OOM during generation.\n"
        "Fixes: use a shorter reference clip, shorten TARGET_TEXT, or restart the runtime."
    )

speech = outputs.speech_outputs[0]   # audio tensor @ 24 kHz
print("✅ Generation done. Output tensor shape:", tuple(speech.shape))

## 8. Save and play output audio
We save with the processor's own `save_audio` (canonical 24 kHz WAV), then read
it back for the in-notebook player so what you hear is exactly what's on disk.

In [ ]:
# Canonical save via the processor (handles tensor shape/scaling correctly).
processor.save_audio(speech, output_path=OUTPUT_PATH)

# Read back for display (and a robust fallback if save_audio is unavailable).
try:
    audio_np, sr = sf.read(OUTPUT_PATH)
except Exception:
    audio_np = speech.detach().float().cpu().numpy().squeeze()
    sr = SAMPLE_RATE
    sf.write(OUTPUT_PATH, audio_np, sr)

audio_np = np.asarray(audio_np, dtype=np.float32).squeeze()
print(f"✅ Saved {OUTPUT_PATH}  ({len(audio_np)/sr:.1f}s @ {sr} Hz)")

from IPython.display import Audio, display
print("🔊 Cloned voice saying the target text (Hindi):")
display(Audio(audio_np, rate=sr))

# In Colab, uncomment to download:
# from google.colab import files; files.download(OUTPUT_PATH)

## 9. Optional extras — multi-speaker Hindi dialogue

VibeVoice's flagship capability is **multi-speaker** long-form synthesis. With a
second reference clip you can generate a two-voice Hindi conversation: label
lines `Speaker 1:` / `Speaker 2:` and pass both clips in order.

> This is a bonus capability beyond single-speaker cloning. Set the flag, point
> `SECOND_REF_PATH` at a **different** clean voice clip, and run.

In [ ]:
# ====== OPTIONAL — 2-speaker Hindi dialogue (run only if needed) ===============
RUN_EXTRAS      = False              # set True to execute this cell's body
SECOND_REF_PATH = ""                 # path to a SECOND, different voice clip

if RUN_EXTRAS:
    if not SECOND_REF_PATH or not os.path.exists(SECOND_REF_PATH):
        raise FileNotFoundError(
            "Set SECOND_REF_PATH to a second clean voice clip (different speaker)."
        )
    # Light-clean the 2nd reference the same way as the 1st.
    w2, _ = load_and_preprocess(SECOND_REF_PATH)
    CLEAN_REF_PATH_2 = "reference_clean_2.wav"
    sf.write(CLEAN_REF_PATH_2, w2, SAMPLE_RATE)

    # A short two-turn Hindi dialogue. Speaker 1 -> clip 1, Speaker 2 -> clip 2.
    dialogue = (
        "Speaker 1: नमस्ते, आज हम VibeVoice के बारे में बात करेंगे।\n"
        "Speaker 2: बहुत अच्छा! यह एक बहु-वक्ता आवाज़ मॉडल है।"
    )

    ins = processor(
        text=[dialogue],
        voice_samples=[[CLEAN_REF_PATH, CLEAN_REF_PATH_2]],   # order = Speaker 1, 2
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )
    try:
        ins = ins.to(DEVICE)
    except Exception:
        ins = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in ins.items()}

    with torch.inference_mode():
        out2 = model.generate(
            **ins, max_new_tokens=None, cfg_scale=CFG_SCALE,
            tokenizer=processor.tokenizer, generation_config={"do_sample": False},
            verbose=True,
        )
    processor.save_audio(out2.speech_outputs[0], output_path="output_dialogue.wav")
    a2, s2 = sf.read("output_dialogue.wav")
    print("✅ Saved output_dialogue.wav (2-speaker Hindi dialogue)")
    display(Audio(np.asarray(a2, dtype=np.float32).squeeze(), rate=s2))
else:
    print("Extras skipped. Set RUN_EXTRAS = True (and SECOND_REF_PATH) to run a dialogue.")

## 10. Troubleshooting notes

| Symptom | Fix |
|---|---|
| **`ModuleNotFoundError: vibevoice`** | Re-run the install cell first. It now injects `REPO_DIR` into `sys.path` and verifies `import vibevoice`; if that still fails, delete the local `VibeVoice/` clone and run the notebook top-to-bottom again. |
| **No GPU detected** | Colab: *Runtime → Change runtime type → GPU*. CPU/MPS work but are slow. |
| **`ffmpeg` errors / can't read MP3** | Install ffmpeg (`!apt-get -y install ffmpeg` on Colab) or convert the sample to WAV first. |
| **Processor load fails on the Hindi repo** | Expected — the notebook falls back to the base `vibevoice/VibeVoice-1.5B` processor automatically. |
| **Model download fails / gated** | `huggingface-cli login` (or set `HF_TOKEN`) and accept the model terms on its HF page. |
| **Attention / flash-attn error** | Keep `ATTN_IMPLEMENTATION='sdpa'` (default). Only use `'flash_attention_2'` on Ampere+ GPUs with flash-attn built. |
| **CUDA out of memory** | Restart the runtime; use a shorter reference / shorter `TARGET_TEXT`; ~8 GB VRAM is needed. |
| **Output is English-sounding / mispronounced** | Keep `TARGET_TEXT` in **Hindi (Devanagari)** — this is a Hindi fine-tune. Re-run with a different `SEED`. |
| **Clone doesn't resemble the sample** | Use a clean single-speaker clip (~10–30 s); raise `CFG_SCALE` (1.3–1.6); try another reference / seed. |
| **Robotic / unstable runs** | Re-run with a different `SEED`; keep the reference clean; ensure it's mono 24 kHz (the prep cell handles this). |

### Going further
- **Multi-speaker:** label lines `Speaker 1:` / `Speaker 2:` (up to 4) and pass
  one reference clip per speaker, in order (see the extras cell).
- **Fine-tuning:** for production-grade, consistent cloning of one speaker, a
  short fine-tune beats zero-shot. See the
  [vibevoice-community/VibeVoice](https://github.com/vibevoice-community/VibeVoice)
  repo and the [Hindi model card](https://huggingface.co/tarun7r/vibevoice-hindi-1.5B).